# Camada Bronze — `frequencia`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Descrição da tabela:** Micro-amostra complementar de frequência em aula (20 registros / 9 alunos)  
**Arquivo bruto:** `Atividade11 - Frequência das Aulas.txt`

In [1]:
import json
import sys
from pathlib import Path

# Garante que notebooks/ esteja no sys.path independente do diretório de execução
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.bronze_utils import (
    get_project_paths,
    ingest_raw_file,
    profile_bronze_table,
    save_bronze_outputs,
)

ROOT_DIR, RAW_DIR, BRONZE_DIR = get_project_paths()
print(f"Origem: {RAW_DIR / 'Atividade11 - Frequência das Aulas.txt'}")
print(f"Destino Bronze: {BRONZE_DIR}")

Origem: C:\Projects\trabalho-final-mineacao-de-dados\Bases TRabalho Final\Atividade11 - Frequência das Aulas.txt
Destino Bronze: C:\Projects\trabalho-final-mineacao-de-dados\data\bronze


In [2]:
df, meta = ingest_raw_file(
    dataset_name='frequencia',
    filename='Atividade11 - Frequência das Aulas.txt',
    sep='|',
    mixed_sep=False,
    raw_dir=RAW_DIR,
)

profile = profile_bronze_table(df, meta)
parquet_path, profile_path = save_bronze_outputs(df, profile, 'frequencia', BRONZE_DIR)

print(f"Linhas ingeridas: {meta['rows']:,} | Tamanho bruto: {meta['size_mb']} MB")
print(f"IDs únicos de aluno: {profile['unique_id_aluno']:,} | Duplicatas exatas: {profile['exact_duplicates']:,}")
print(f"Salvo em: {parquet_path.name} e {profile_path.name}")
df.head(5)

Linhas ingeridas: 20 | Tamanho bruto: 0.0017 MB
IDs únicos de aluno: 9 | Duplicatas exatas: 0
Salvo em: frequencia.parquet e frequencia_profile.json


,id_frequencia,id_aluno,data_aula,curso,disciplina,turno,situacao,minutos_atraso,_bronze_source_file,_bronze_ingested_at
0,FREQ000000001,10245,2024-02-15,Engenharia de Software,Algoritmos e Programação,Matutino,Presente,0,Atividade11 - Frequência das Aulas.txt,2026-11-13T18:54:14Z
1,FREQ000000002,83921,2024-02-15,Administração,Gestão Estratégica,Noturno,Atrasado,15,Atividade11 - Frequência das Aulas.txt,2026-11-13T18:54:14Z
2,FREQ000000003,45210,2024-02-15,Direito,Direito Constitucional,Vespertino,Ausente,0,Atividade11 - Frequência das Aulas.txt,2026-11-13T18:54:14Z
3,FREQ000000004,12093,2024-02-15,Medicina,Anatomia Humana,Matutino,Presente,0,Atividade11 - Frequência das Aulas.txt,2026-11-13T18:54:14Z
4,FREQ000000005,73104,2024-02-15,Engenharia de Software,Bancos de Dados,Noturno,Atrasado,8,Atividade11 - Frequência das Aulas.txt,2026-11-13T18:54:14Z


In [3]:
print("=== Campos vazios por coluna ===")
print(json.dumps(profile["empty_counts"], ensure_ascii=False, indent=2))
if profile["anomalies"]:
    print("\n=== Anomalias detectadas para tratamento na Silver ===")
    print(json.dumps(profile["anomalies"], ensure_ascii=False, indent=2))

=== Campos vazios por coluna ===
{
  "id_frequencia": 0,
  "id_aluno": 0,
  "data_aula": 0,
  "curso": 0,
  "disciplina": 0,
  "turno": 0,
  "situacao": 0,
  "minutos_atraso": 0
}

=== Anomalias detectadas para tratamento na Silver ===
{
  "situacao_distribuicao": {
    "Presente": 11,
    "Atrasado": 5,
    "Ausente": 4
  },
  "escopo_observacao": "Micro-amostra complementar (20 linhas / 9 alunos) — não extrapolar para 1M."
}
